# 06 · Validate Silver

**Layer:** Silver &nbsp;|&nbsp; **Stage:** data quality gate &nbsp;|&nbsp; **Target:** `banking_silver.dq_results`

The fan-in point for the two parallel silver builds, and the second quality
gate of the pipeline. Where the bronze gate asked *"did we land plausible
data?"*, this one asks *"did our transformations actually fix it?"* — so the
thresholds are strict rather than tolerant. A duplicate that survives into
silver is a bug in `05_build_silver_txn_clean`, not a bad source file.

**Checks performed**

- both silver tables are populated
- `dim_account.account_id` is unique and never null (strict, threshold 1.0)
- `txn_clean.txn_id` is unique (strict, threshold 1.0)
- zero orphan transactions — referential integrity must be total after the join
- every amount is positive and every currency normalised
- `amount_inr` is populated for every row, so gold can aggregate safely
- reconciliation: silver row count + reject count accounts for all of bronze


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("medallion-validate-silver").enableHiveSupport().getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "16")

RUN_DATE  = "2026-09-03"
BATCH_ID  = f"batch_{RUN_DATE.replace('-', '')}"
BRONZE_DB = "banking_bronze"
SILVER_DB = "banking_silver"

STRICT = 1.0     # post-transformation checks admit no failures
LENIENT = 0.95

dim  = spark.table(f"{SILVER_DB}.dim_account")
txn  = spark.table(f"{SILVER_DB}.txn_clean")
rej  = spark.table(f"{SILVER_DB}.txn_rejects")

dim_total = dim.count()
txn_total = txn.count()
rej_total = rej.count()
print(f"dim_account: {dim_total}   txn_clean: {txn_total}   txn_rejects: {rej_total}")

In [ ]:
checks = []

def record(check_name, table, observed, threshold, passed, detail=""):
    checks.append((
        BATCH_ID, RUN_DATE, table, check_name,
        float(observed), float(threshold),
        "PASS" if passed else "FAIL", detail,
    ))
    flag = "PASS" if passed else "FAIL"
    print(f"  [{flag}] {table:<14} {check_name:<30} observed={observed:.4f} threshold={threshold}")

In [ ]:
print("volume checks")

record("table_not_empty", "dim_account", dim_total, 1, dim_total > 0, f"{dim_total} rows")
record("table_not_empty", "txn_clean",  txn_total, 1, txn_total > 0, f"{txn_total} rows")

In [ ]:
print("key integrity checks (strict)")

dim_distinct = dim.select("account_id").distinct().count()
dim_uniq = dim_distinct / dim_total if dim_total else 0.0
record("account_id_unique", "dim_account", dim_uniq, STRICT, dim_uniq >= STRICT,
       f"{dim_total - dim_distinct} duplicate account_id survived dedupe")

dim_nulls = dim.filter(F.col("account_id").isNull()).count()
record("account_id_not_null", "dim_account", 1.0 if dim_nulls == 0 else 0.0, STRICT,
       dim_nulls == 0, f"{dim_nulls} null account_id")

txn_distinct = txn.select("txn_id").distinct().count()
txn_uniq = txn_distinct / txn_total if txn_total else 0.0
record("txn_id_unique", "txn_clean", txn_uniq, STRICT, txn_uniq >= STRICT,
       f"{txn_total - txn_distinct} duplicate txn_id survived dedupe")

In [ ]:
print("referential integrity (strict)")

orphans = txn.join(dim.select("account_id"), on="account_id", how="left_anti").count()
orphan_ratio = 1.0 - (orphans / txn_total if txn_total else 0.0)
record("no_orphan_transactions", "txn_clean", orphan_ratio, STRICT, orphans == 0,
       f"{orphans} transaction(s) with no matching account")

In [ ]:
print("value checks")

neg = txn.filter(F.col("amount_inr") <= 0).count()
record("amount_inr_positive", "txn_clean", 1.0 if neg == 0 else 0.0, STRICT, neg == 0,
       f"{neg} row(s) with amount_inr <= 0")

null_fx = txn.filter(F.col("amount_inr").isNull() | F.col("fx_rate_to_inr").isNull()).count()
record("fx_conversion_complete", "txn_clean", 1.0 if null_fx == 0 else 0.0, STRICT, null_fx == 0,
       f"{null_fx} row(s) missing an FX conversion")

null_date = txn.filter(F.col("txn_date").isNull()).count()
date_ratio = 1.0 - (null_date / txn_total if txn_total else 0.0)
record("txn_date_parsed", "txn_clean", date_ratio, STRICT, null_date == 0,
       f"{null_date} row(s) with an unparseable timestamp")

In [ ]:
print("reconciliation")

bronze_total = spark.table(f"{BRONZE_DB}.raw_transactions").count()
bronze_distinct = (
    spark.table(f"{BRONZE_DB}.raw_transactions")
    .filter(F.col("txn_id").isNotNull())
    .select("txn_id").distinct().count()
)
accounted = txn_total + rej_total
recon_ratio = accounted / bronze_distinct if bronze_distinct else 0.0

record("row_reconciliation", "txn_clean", recon_ratio, LENIENT, recon_ratio >= LENIENT,
       f"{accounted} accounted for out of {bronze_distinct} distinct bronze transactions")

In [ ]:
dq = spark.createDataFrame(
    checks,
    "batch_id string, run_date string, table_name string, check_name string, "
    "observed double, threshold double, status string, detail string",
).withColumn("checked_at", F.current_timestamp())

(
    dq.write.mode("overwrite").format("parquet")
    .saveAsTable(f"{SILVER_DB}.dq_results")
)

dq.select("table_name", "check_name", "observed", "threshold", "status", "detail") \
  .orderBy("table_name", "check_name").show(50, truncate=False)

In [ ]:
failed = [c for c in checks if c[6] == "FAIL"]
DQ_STATUS = "FAIL" if failed else "PASS"

print("=" * 60)
print(f"checks run    : {len(checks)}")
print(f"checks failed : {len(failed)}")
for c in failed:
    print(f"    - {c[2]}.{c[3]}: {c[7]}")
print(f"DQ_STATUS={DQ_STATUS}")
print("=" * 60)

print({"dq_status": DQ_STATUS, "failed_checks": len(failed), "layer": "silver"})